# AMSL Agentic AI Live API Study

이 노트북은 웹사이트에서 배운 내용을 직접 실행해 보는 **실행 노트북**입니다. 교재는 웹사이트이고, 이 파일은 코드를 눌러 보며 확인하는 실습장입니다.

처음부터 모든 용어를 알 필요는 없습니다. 먼저 웹사이트의 `학습 섹션`에서 Section 0을 읽고, 여기서는 안내된 셀을 위에서 아래로 실행하면 됩니다.

웹사이트: https://intern-study.donghyunlee.me

진행 순서:

1. 웹사이트의 `학습 섹션`에서 Section 0을 읽습니다.
2. 이 노트북에서 Section 0 셀을 실행해 패키지, API key, 첫 연결을 확인합니다.
3. Section 1-4 셀을 순서대로 실행합니다.
4. 마지막에 사용량을 확인합니다.

Section 5 Linux/WSL2는 노트북 셀이 아니라 웹사이트에서 따로 보는 개발 환경 섹션입니다.

> 실행 전 확인: 이 노트북은 실제 OpenAI API를 호출합니다. `Run All`을 누르면 happy path 기준 약 12회 정도의 API 요청이 발생할 수 있고, JSON 수정 재시도 때문에 더 늘 수 있습니다. 처음에는 반드시 웹사이트의 `API 키와 비용` 설명을 읽고, 셀을 위에서 아래로 하나씩 실행하세요.

## Section 0: 패키지 준비

웹사이트의 `학습 섹션` Section 0을 읽은 뒤 실행합니다. 처음 실행하는 환경에서는 아래 셀을 실행하세요. 이미 `uv run --python 3.12 --with-requirements examples/requirements.txt --with notebook ...` 명령으로 Jupyter를 열었다면 빠르게 넘어갈 수 있습니다.

노트북 안의 설치 셀은 `examples/requirements.txt`를 읽어서 현재 노트북 커널에 필요한 패키지를 맞춥니다. 패키지 목록은 requirements 파일 하나만 기준으로 관리합니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

requirements_candidates = [
    Path("examples/requirements.txt"),
    Path("../examples/requirements.txt"),
]
requirements = next((path for path in requirements_candidates if path.exists()), None)
if requirements is None:
    raise FileNotFoundError("examples/requirements.txt를 찾지 못했습니다. 노트북을 amsl-internship-study 폴더 구조 안에서 실행하세요.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
print(f"installed requirements from {requirements}")

## Section 0: API key와 모델 설정

웹사이트의 `API 키와 비용` 설명을 먼저 읽습니다. API key는 웹사이트에 입력하지 않고, 같은 폴더 또는 프로젝트의 `examples/.env`에 아래처럼 적습니다.

```bash
OPENAI_API_KEY=...
OPENAI_MODEL=gpt-5.4-mini
```

`gpt-5.4-mini`는 현재 검증된 수업용 기본 모델명입니다. 수업에서 다른 모델명을 안내받으면 `OPENAI_MODEL`만 바꿉니다.

In [ ]:
from __future__ import annotations

import getpass
import json
import os
import re
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

# Load .env from common locations.
for candidate in [Path.cwd() / ".env", Path.cwd() / "examples" / ".env", Path.cwd().parent / "examples" / ".env"]:
    if candidate.exists():
        load_dotenv(candidate)
        print(f"Loaded env from {candidate}")
        break
else:
    load_dotenv(dotenv_path=Path(".env"), override=False)

if not os.getenv("OPENAI_API_KEY"):
    key = getpass.getpass("OpenAI API key: ")
    if key.strip():
        os.environ["OPENAI_API_KEY"] = key.strip()

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client: OpenAI | None = None
usage_log: list[dict[str, int | str | None]] = []

print("Model:", MODEL)


## Section 0: 공통 helper

아래 helper는 이후 Section 1-4에서 공통으로 사용합니다. `call_text`와 `call_json`은 OpenAI Responses API를 호출합니다. `call_json`은 모델 출력에서 JSON object를 파싱하고, 필요하면 한 번 더 수정 요청을 보낼 수 있습니다.

In [ ]:
def record_usage(label: str, response: Any) -> None:
    usage = getattr(response, "usage", None)
    usage_log.append(
        {
            "label": label,
            "input_tokens": getattr(usage, "input_tokens", None),
            "output_tokens": getattr(usage, "output_tokens", None),
            "total_tokens": getattr(usage, "total_tokens", None),
        }
    )


def get_client() -> OpenAI:
    global client
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY가 없습니다. 웹사이트의 Section 0 안내를 보고 .env 파일을 만들거나 "
            "노트북의 key 입력 프롬프트에 수업용 API key를 입력하세요."
        )
    if client is None:
        client = OpenAI()
    return client


def call_text(prompt: str, label: str) -> str:
    response = get_client().responses.create(model=MODEL, input=prompt, max_output_tokens=700)
    record_usage(label, response)
    return response.output_text.strip()


def parse_json_object(text: str) -> dict[str, Any]:
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        value = json.loads(text[start : end + 1])
    if not isinstance(value, dict):
        raise ValueError("Expected a JSON object.")
    return value


def call_json(prompt: str, label: str, max_attempts: int = 2) -> dict[str, Any]:
    json_prompt = prompt + "\n\nReturn only one valid JSON object. Do not use Markdown fences."
    last_error: Exception | None = None
    current_prompt = json_prompt
    for _ in range(max_attempts):
        text = call_text(current_prompt, label=label)
        try:
            return parse_json_object(text)
        except (json.JSONDecodeError, ValueError) as error:
            last_error = error
            current_prompt = (
                json_prompt
                + "\n\nYour previous response was not valid JSON. "
                + f"Validation error: {error}. Return only corrected JSON."
            )
    raise ValueError(f"Model did not return valid JSON after {max_attempts} attempts.") from last_error


## Section 0: 첫 연결 테스트(smoke test)

웹사이트의 Section 0과 `API 키와 비용`을 읽은 뒤 실행합니다. 이 셀은 학습 내용 평가가 아니라 API key, 모델명, 네트워크, 패키지 설치가 맞는지 확인하는 셀입니다.

In [ ]:
print(call_text("한국어 한 문장으로 OpenAI API 연결이 정상이라고 답하세요.", label="smoke_test"))


## Section 0: 답변 형식 검사 준비

AI 출력은 바로 믿지 않고 정해진 형식을 지키는지 검사합니다. 여기서는 그 검사를 위해 Pydantic이라는 Python 도구를 사용합니다. Section 1-4에서 계속 사용할 공통 schema를 먼저 정의합니다.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)


class Evidence(StrictModel):
    source_id: str = Field(min_length=1)
    quote: str = Field(min_length=1)


def normalize_evidence_items(
    value: Any,
    *,
    default_source_id: str = "not specified",
    fallback: list[Evidence] | None = None,
) -> list[Evidence]:
    """Convert common LLM evidence variants into Evidence objects."""
    if value is None:
        return []
    if isinstance(value, dict):
        value = [value]
    if not isinstance(value, list):
        value = [value]

    normalized: list[Evidence] = []
    for item in value:
        if isinstance(item, Evidence):
            normalized.append(item)
        elif isinstance(item, dict):
            source_id = str(item.get("source_id") or default_source_id)
            quote = str(item.get("quote") or item.get("text") or "").strip()
            if quote:
                normalized.append(Evidence(source_id=source_id, quote=quote))
        elif isinstance(item, str):
            quote = item.strip()
            if not quote:
                continue
            matched = None
            for candidate in fallback or []:
                if quote in candidate.quote or candidate.quote in quote:
                    matched = candidate.source_id
                    break
            normalized.append(Evidence(source_id=matched or default_source_id, quote=quote))
    return normalized


def filter_evidence_to_allowed(
    value: Any,
    allowed: list[Evidence],
    *,
    default_source_id: str = "not specified",
) -> list[Evidence]:
    """Keep only evidence that can be matched to the retrieved context."""
    candidates = normalize_evidence_items(
        value,
        default_source_id=default_source_id,
        fallback=allowed,
    )
    accepted: list[Evidence] = []
    seen: set[tuple[str, str]] = set()

    for candidate in candidates:
        for item in allowed:
            same_source = candidate.source_id == item.source_id
            exact_quote = candidate.quote == item.quote
            quote_inside_allowed = candidate.quote in item.quote
            allowed_inside_quote = item.quote in candidate.quote
            if same_source and (exact_quote or quote_inside_allowed or allowed_inside_quote):
                key = (item.source_id, item.quote)
                if key not in seen:
                    accepted.append(item)
                    seen.add(key)
                break

    return accepted


class DocumentExtraction(StrictModel):
    topic: str = Field(min_length=1)
    objective: str = Field(min_length=1)
    method: str = Field(min_length=1)
    result: str = Field(min_length=1)
    limitations: list[str] = Field(default_factory=list)
    evidence: list[Evidence] = Field(default_factory=list)
    confidence: float = Field(ge=0.0, le=1.0)

    @field_validator("limitations", mode="before")
    @classmethod
    def normalize_limitations(cls, value: Any) -> list[str]:
        """Accept common LLM variants while keeping the public schema stable."""
        if value is None:
            return []
        if isinstance(value, list):
            return [str(item) for item in value if str(item).strip()]
        if isinstance(value, str):
            normalized = value.strip()
            if not normalized or normalized.lower() in {"none", "not specified", "n/a", "없음"}:
                return []
            return [normalized]
        return [str(value)]

    @field_validator("confidence", mode="before")
    @classmethod
    def reject_non_numeric_confidence(cls, value: Any) -> float:
        if isinstance(value, bool) or not isinstance(value, int | float):
            raise ValueError("confidence must be a number between 0 and 1")
        return float(value)

    @model_validator(mode="after")
    def require_evidence_for_confident_claims(self) -> "DocumentExtraction":
        if self.confidence >= 0.5 and not self.evidence:
            raise ValueError("evidence is required when confidence is 0.5 or higher")
        return self

class GroundedAnswer(StrictModel):
    answer_ko: str = Field(min_length=1)
    evidence: list[Evidence] = Field(default_factory=list)
    source_ids: list[str] = Field(default_factory=list)
    not_answered: bool = False


class WorkflowAnswer(StrictModel):
    answer_ko: str = Field(min_length=1)
    facts: DocumentExtraction
    evidence: list[Evidence] = Field(default_factory=list)
    not_answered: bool = False
    log: list[str] = Field(default_factory=list)


## Section 1: AI에게 요청 보내기

먼저 웹사이트의 `학습 섹션`에서 Section 1 설명과 핵심 개념을 읽습니다. 그 다음 이 셀에서 짧은 문서의 `topic`, `objective`, `method`, `result`, `limitations`, `evidence`, `confidence`를 추출합니다.

실행한 뒤 웹사이트의 `결과 확인` 탭과 비교해 구조, evidence, 실패 처리 정책이 맞는지 확인합니다.

In [ ]:
NOTES = [
    {
        "id": "note_a",
        "text": "A support operations team piloted a ticket-routing assistant for incoming customer requests. The assistant reads a ticket, identifies the affected product area, assigns a confidence score, and routes uncertain cases to human review. In the pilot queue, manual triage work decreased by 32 percent while escalation errors were tracked for weekly review.",
    },
    {
        "id": "note_b",
        "text": "A documentation team built a retrieval-augmented question answering workflow for internal API guides. The system indexes short documentation chunks, retrieves relevant passages for each question, and requires every answer to include source identifiers. The first evaluation showed better answer traceability, but several broad questions still retrieved irrelevant chunks.",
    },
    {
        "id": "note_c",
        "text": "An engineering productivity group created a release note assistant that reads merged pull request summaries and groups them by audience. The workflow drafts user-facing notes, flags items without issue IDs, and asks a maintainer to approve the final message before publication.",
    },
    {
        "id": "note_d",
        "text": "A team announced an AI automation project that improves operational efficiency. The announcement says the system outperforms the previous process, but it does not specify the input data, evaluation metric, or validation procedure.",
    },
]


def calibrated_confidence(raw_confidence: float, raw: dict) -> float:
    confidence = float(raw_confidence)
    joined = " ".join(
        str(value)
        for key in ("objective", "method", "result", "limitations")
        for value in ([raw.get(key)] if not isinstance(raw.get(key), list) else raw.get(key, []))
    ).lower()
    missing_markers = (
        "not specified",
        "does not specify",
        "unspecified",
        "missing",
        "lack",
        "lacked",
        "없음",
        "명시되지",
    )
    if any(marker in joined for marker in missing_markers):
        return min(confidence, 0.45)
    return confidence


def extract_note(note: dict[str, str]) -> DocumentExtraction:
    raw = call_json(
        f"""
당신은 기술 문서를 구조화하는 개발 보조자입니다.
아래 문서에서 정보를 추출하세요.

필드:
- topic
- objective
- method
- result
- limitations: 한계 목록. 없으면 빈 배열
- evidence: source_id와 quote를 가진 배열. source_id는 \"{note['id']}\"만 사용
- confidence: 0과 1 사이 숫자

규칙:
- 문서에 없는 내용은 추측하지 말고 \"not specified\"라고 쓰세요.
- evidence quote는 반드시 아래 문서에서 온 문장이나 구절이어야 합니다.
- 입력 데이터, 평가 지표, 검증 방법처럼 핵심 정보가 명시되지 않았으면 confidence를 0.45 이하로 두세요.

문서:
{note['text']}
""".strip(),
        label=f"extract_{note['id']}",
    )
    return DocumentExtraction(
        topic=raw["topic"],
        objective=raw["objective"],
        method=raw["method"],
        result=raw["result"],
        limitations=raw.get("limitations", []),
        evidence=filter_evidence_to_allowed(raw.get("evidence", []), [Evidence(source_id=note["id"], quote=note["text"])], default_source_id=note["id"]),
        confidence=calibrated_confidence(raw["confidence"], raw),
    )


extractions = [extract_note(note) for note in NOTES]
for extraction in extractions:
    print(extraction.model_dump_json(indent=2))


## Section 2: 답변을 정해진 형식으로 받기

먼저 웹사이트에서 AI 답변을 정해진 형식으로 받는 설명을 읽습니다. 이 셀은 prompt와 출력 확인 도구를 연결해, AI 답변이 코드에서 사용할 수 있는 구조로 바뀌는지 확인하는 실행 실습입니다.

실행한 뒤 웹사이트의 `결과 확인` 탭과 비교해 구조, evidence, 실패 처리 정책이 맞는지 확인합니다.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

parser = PydanticOutputParser(pydantic_object=DocumentExtraction)
prompt_template = PromptTemplate(
    template=(
        "당신은 기술 문서를 읽는 개발 보조자입니다.\n"
        "문서에서 topic, objective, method, result, limitations, evidence, confidence를 추출하세요.\n"
        "문서에 없는 정보는 추측하지 말고 not specified로 쓰세요.\n\n"
        "evidence의 source_id는 반드시 입력 문서 ID인 {source_id}만 사용하세요.\n\n"
        "{format_instructions}\n\n"
        "문서:\n{note}"
    ),
    input_variables=["source_id", "note"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

item = NOTES[0]
prompt_text = prompt_template.format(source_id=item["id"], note=item["text"])
model_text = call_text(
    prompt_text + "\n\nReturn only one valid JSON object. Do not use Markdown fences.",
    label="langchain_parser_contract",
)
parsed_raw = parser.parse(model_text)
parsed = DocumentExtraction(
    topic=parsed_raw.topic,
    objective=parsed_raw.objective,
    method=parsed_raw.method,
    result=parsed_raw.result,
    limitations=parsed_raw.limitations,
    evidence=filter_evidence_to_allowed(
        parsed_raw.evidence,
        [Evidence(source_id=item["id"], quote=item["text"])],
        default_source_id=item["id"],
    ),
    confidence=parsed_raw.confidence,
)
print(parsed.model_dump_json(indent=2))


## Section 3: 문서 근거로 답하게 하기

먼저 웹사이트에서 문서에서 근거를 찾아 답하는 설명을 읽습니다. 이 셀에서는 필요한 문서를 단어 기반으로 간단히 찾고, 실제 AI 응답은 찾은 문서 근거 안에서 만들도록 요청합니다. 실제 프로젝트에서는 문서를 찾는 부분을 더 강한 검색 방식으로 바꿀 수 있습니다.

실행한 뒤 웹사이트의 `결과 확인` 탭과 비교해 구조, evidence, 실패 처리 정책이 맞는지 확인합니다.

In [ ]:
CORPUS = [
    {
        "source_id": "doc_001_support_triage",
        "text": "Support triage assistants classify incoming tickets, identify the affected product area, and route each ticket to the right team. A safe triage workflow should include evidence from the ticket text, a confidence score, and a fallback path for uncertain cases.",
    },
    {
        "source_id": "doc_002_api_onboarding",
        "text": "API onboarding guides should explain authentication, environment variables, rate limits, and a minimal smoke test. API keys must be stored outside source code, and the first live request should use a low-risk prompt that checks connectivity.",
    },
    {
        "source_id": "doc_003_rag_quality",
        "text": "Retrieval-augmented generation (RAG) systems should inspect retrieved chunks before trusting an answer. If the retrieved context does not contain the answer, the system should return not_answered instead of inventing details.",
    },
    {
        "source_id": "doc_004_release_notes",
        "text": "Release note generators should group changes by audience and cite pull requests or issue IDs for each claim. If the source note does not support a statement, the assistant should not claim that the feature shipped.",
    },
]

STOPWORDS = {"은", "는", "이", "가", "을", "를", "에", "의", "와", "과", "the", "a", "an", "and", "or", "to", "for", "of", "in"}


def keyword_tokens(text: str) -> set[str]:
    raw = re.findall(r"[A-Za-z0-9_.-]+|[가-힣]+", text.lower())
    tokens = set()
    for token in raw:
        for suffix in ["에서는", "에는", "으로", "에서", "에게", "까지", "부터", "처럼", "보다", "하고", "해야", "하나", "인가", "은", "는", "이", "가", "을", "를"]:
            if token.endswith(suffix) and len(token) > len(suffix) + 1:
                token = token[: -len(suffix)]
                break
        if token and token not in STOPWORDS:
            tokens.add(token)
    return tokens


def retrieve(query: str, top_k: int = 2) -> list[Evidence]:
    q = keyword_tokens(query)
    ranked = sorted(CORPUS, key=lambda doc: len(q & keyword_tokens(doc["text"])), reverse=True)
    selected = [doc for doc in ranked[:top_k] if len(q & keyword_tokens(doc["text"])) > 0]
    return [Evidence(source_id=doc["source_id"], quote=doc["text"]) for doc in selected]


def answer_with_rag(question: str) -> GroundedAnswer:
    evidence = retrieve(question)
    if not evidence:
        return GroundedAnswer(answer_ko="제공된 문서에서 답을 찾을 수 없습니다.", not_answered=True)
    raw = call_json(
        f"""
당신은 RAG 답변 생성기입니다.
질문과 검색된 evidence만 사용해 한국어로 답하세요.

규칙:
- evidence에 답이 없으면 not_answered를 true로 두세요.
- evidence에 없는 내용을 추측하지 마세요.
- evidence 배열에는 실제 사용한 source_id와 quote만 남기세요.
- source_ids는 evidence의 source_id 목록이어야 합니다.

질문:
{question}

evidence:
{json.dumps([item.model_dump() for item in evidence], ensure_ascii=False, indent=2)}

출력 JSON 필드:
answer_ko, evidence, source_ids, not_answered
""".strip(),
        label="rag_answer",
    )
    accepted_evidence = filter_evidence_to_allowed(raw.get("evidence", []), evidence)
    not_answered = bool(raw.get("not_answered", False)) or not accepted_evidence
    if not_answered:
        return GroundedAnswer(
            answer_ko="제공된 문서에서 답을 찾을 수 없습니다.",
            evidence=[],
            source_ids=[],
            not_answered=True,
        )
    return GroundedAnswer(
        answer_ko=raw["answer_ko"],
        evidence=accepted_evidence,
        source_ids=[item.source_id for item in accepted_evidence],
        not_answered=False,
    )


questions = [
    "API onboarding guide에는 무엇이 들어가야 해?",
    "support ticket triage는 어떤 fallback을 가져야 해?",
    "RAG 답변을 신뢰하기 전에 무엇을 확인해야 해?",
    "pricing 정보는 어디에 있어?",
]
rag_answers = {question: answer_with_rag(question) for question in questions}
for question, answer in rag_answers.items():
    print("\nQUESTION:", question)
    print(answer.model_dump_json(indent=2))


## Section 4: 여러 단계를 하나로 연결하기

먼저 웹사이트에서 여러 실행 단계를 연결하는 방식과 완전 자율 AI 도우미의 차이를 읽습니다. 이 셀에서는 문서 근거 답변에서 다시 필요한 사실을 뽑고, 최종 답변이 정해진 형식을 지키는지 확인합니다.

실행한 뒤 웹사이트의 `결과 확인` 탭과 비교해 구조, evidence, 실패 처리 정책이 맞는지 확인합니다.

In [ ]:
def build_facts_from_answer(answer: GroundedAnswer) -> DocumentExtraction:
    if answer.not_answered:
        return DocumentExtraction(
            topic="not specified",
            objective="not specified",
            method="not specified",
            result="not specified",
            limitations=["no supporting evidence in retrieved context"],
            evidence=answer.evidence,
            confidence=0.2,
        )
    raw = call_json(
        f"""
당신은 workflow의 structured facts 추출 단계입니다.
아래 한국어 답변과 evidence만 사용해 구조화 정보를 추출하세요.

규칙:
- evidence에 없는 내용은 추측하지 말고 \"not specified\"라고 쓰세요.
- evidence quote는 입력 evidence에서 온 문장만 사용하세요.
- confidence는 0과 1 사이 숫자입니다.

answer_ko:
{answer.answer_ko}

evidence:
{json.dumps([item.model_dump() for item in answer.evidence], ensure_ascii=False, indent=2)}

출력 JSON 필드:
topic, objective, method, result, limitations, evidence, confidence
""".strip(),
        label="workflow_facts",
    )
    accepted_evidence = filter_evidence_to_allowed(raw.get("evidence", []), answer.evidence)
    if not accepted_evidence:
        return DocumentExtraction(
            topic="not specified",
            objective="not specified",
            method="not specified",
            result="not specified",
            limitations=["structured extraction did not return supported evidence"],
            evidence=[],
            confidence=0.2,
        )
    return DocumentExtraction(
        topic=raw["topic"],
        objective=raw["objective"],
        method=raw["method"],
        result=raw["result"],
        limitations=raw.get("limitations", []),
        evidence=accepted_evidence,
        confidence=raw["confidence"],
    )


def run_workflow(question: str) -> WorkflowAnswer:
    answer = answer_with_rag(question)
    facts = build_facts_from_answer(answer)
    return WorkflowAnswer(
        answer_ko=answer.answer_ko,
        facts=facts,
        evidence=answer.evidence,
        not_answered=answer.not_answered,
        log=["received question", "retrieved evidence", "generated answer", "validated structured facts"],
    )


for question in ["API onboarding guide에는 무엇이 들어가야 해?", "pricing 정보는 어디에 있어?"]:
    result = run_workflow(question)
    print("\nQUESTION:", question)
    print(result.model_dump_json(indent=2))


## Appendix: 얕은 도메인 브릿지

기술 학습이 핵심이지만, 실제 연구 문서에 적응하기 위해 얕은 도메인 예시를 하나 넣어볼 수 있습니다. 여기서 중요한 것은 화학/재료 이론 암기가 아니라, 낯선 문서도 evidence 기반으로 구조화하는 것입니다.

In [ ]:
domain_note = {
    "id": "domain_note_001",
    "text": "A screening workflow ranked MOF candidates by predicted CO2 adsorption capacity. The pipeline combined literature-derived descriptors with a regression model and flagged candidates that lacked stability measurements.",
}

domain_extraction = extract_note(domain_note)
print(domain_extraction.model_dump_json(indent=2))


## 마무리: 토큰 사용량 확인

웹사이트의 `API 키와 비용` 설명을 읽은 뒤 확인합니다. 아래 셀은 이번 노트북 실행 중 기록된 토큰 사용량을 합산합니다. 실제 비용은 사용한 모델의 최신 가격표에 따라 달라집니다. 문서의 단가 예시는 계산 방법을 보여주기 위한 가상의 예시입니다.

In [ ]:
for row in usage_log:
    print(row)

total_input = sum(row["input_tokens"] or 0 for row in usage_log)
total_output = sum(row["output_tokens"] or 0 for row in usage_log)
total = sum(row["total_tokens"] or 0 for row in usage_log)
print("\nTotal input tokens:", total_input)
print("Total output tokens:", total_output)
print("Total tokens:", total)
print("\n금액은 선택한 OPENAI_MODEL의 최신 input/output token 단가를 곱해서 계산하세요.")
